# Treinamento da SqueezeNet no HEp-2

Cada cenário tem sua própria célula de treinamento. Execute e repita apenas o cenário que quiser; cada execução treina um modelo, avalia no conjunto de teste e imprime as métricas. Os pesos são salvos em `results/` com os parâmetros no nome do arquivo.

A arquitetura é a SqueezeNet 1.0, treinada do zero para seis classes. No Colab, selecione uma GPU antes de começar.

## Preparar o ambiente

Execute uma vez no início do runtime. O dataset e os scripts são obtidos do repositório.

In [1]:
from pathlib import Path
import subprocess
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_DIR = Path("/content/Projeto-PDI")
    if not (PROJECT_DIR / "main_squeezenet.py").is_file():
        PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/orleoncio/Projeto-PDI.git", str(PROJECT_DIR)],
            check=True,
        )
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_DIR = next(
        (path for path in candidates if (path / "main_squeezenet.py").is_file()),
        None,
    )
    if PROJECT_DIR is None:
        raise FileNotFoundError("Abra este notebook no repositório Projeto-PDI ou execute-o no Google Colab.")

import importlib.util
if not IN_COLAB and any(importlib.util.find_spec(package) is None for package in ("torch", "torchvision")):
    subprocess.run([sys.executable, "-m", "pip", "install", "torch", "torchvision"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "pandas", "pillow"],
    check=True,
)

%cd {PROJECT_DIR}
!nvidia-smi

import torch
import torchvision

print("PyTorch:", torch.__version__, "TorchVision:", torchvision.__version__)
assert torch.cuda.is_available(), "Ative uma GPU em Ambiente de execução > Alterar tipo de ambiente de execução."

  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached nvidia_nccl_cu13-2.30.7-py3-none-manylinux_2_18_x86_64.whl.metadata (2.1 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 MB 6.3 MB/s eta 0:00:000m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 6.4 MB/s eta 0:00:000m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 17.2 MB/s eta 0:00:00m eta 0:00:010:01:01
Using cached nvidia_nccl_cu13-2.30.7-py3-none-manylinux_2_18_x86_64.whl (216.0 MB)
   ━━━━

## Parâmetros comuns

Altere os valores abaixo antes de executar os cenários. O LR do SGD pode ser ajustado independentemente.

In [2]:
EPOCHS = 60
BATCH_SIZE = 32
SEED = 99
WEIGHT_DECAY = 0.0002
SGD_LR = 0.003

## Imagens originais

### Original + Adam

In [3]:
!python -u main_squeezenet.py --version 1.0 --preprocessing none --optimizer adam --lr 0.001 --weight-decay {WEIGHT_DECAY} --seed {SEED} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --amp --num-workers 2

Dispositivo: cuda | Arquitetura: SqueezeNet 1.0 (do zero) | Pré-processamento: none | Otimizador: ADAM | Épocas: 60 | Batch size: 32 | LR: 0.001 | Weight decay: 0.0002 | K-fold: desativado | Validação: 10% | AMP: True
Removendo 1 imagem(ns) duplicada(s) do treino que também estão no teste.
Imagens de treinamento: 9788
Imagens de validação:   1088
Imagens de teste final: 2720
Epoch [1/60] Loss: 1.3278 Train Acc: 45.15% | Val Acc: 53.12% ★ [BEST]
Epoch [2/60] Loss: 1.0071 Train Acc: 63.20% | Val Acc: 71.23% ★ [BEST]
Epoch [3/60] Loss: 0.8310 Train Acc: 71.34% | Val Acc: 67.46%
Epoch [4/60] Loss: 0.7408 Train Acc: 75.03% | Val Acc: 75.46% ★ [BEST]
Epoch [5/60] Loss: 0.6276 Train Acc: 78.00% | Val Acc: 79.32% ★ [BEST]
Epoch [6/60] Loss: 0.5322 Train Acc: 81.11% | Val Acc: 81.34% ★ [BEST]
Epoch [7/60] Loss: 0.4823 Train Acc: 82.86% | Val Acc: 84.10% ★ [BEST]
Epoch [8/60] Loss: 0.4803 Train Acc: 82.86% | Val Acc: 82.63%
Epoch [9/60] Loss: 0.4411 Train Acc: 84.56% | Val Acc: 86.49% ★ [BEST]
E

### Original + SGD

In [4]:
!python -u main_squeezenet.py --version 1.0 --preprocessing none --optimizer sgd --lr {SGD_LR} --weight-decay {WEIGHT_DECAY} --seed {SEED} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --amp --num-workers 2

Dispositivo: cuda | Arquitetura: SqueezeNet 1.0 (do zero) | Pré-processamento: none | Otimizador: SGD | Épocas: 60 | Batch size: 32 | LR: 0.003 | Weight decay: 0.0002 | K-fold: desativado | Validação: 10% | AMP: True
Removendo 1 imagem(ns) duplicada(s) do treino que também estão no teste.
Imagens de treinamento: 9788
Imagens de validação:   1088
Imagens de teste final: 2720
Epoch [1/60] Loss: 1.4942 Train Acc: 34.99% | Val Acc: 38.97% ★ [BEST]
Epoch [2/60] Loss: 1.2201 Train Acc: 49.50% | Val Acc: 58.18% ★ [BEST]
Epoch [3/60] Loss: 1.0635 Train Acc: 59.53% | Val Acc: 58.18%
Epoch [4/60] Loss: 0.9737 Train Acc: 63.14% | Val Acc: 64.80% ★ [BEST]
Epoch [5/60] Loss: 0.8380 Train Acc: 69.31% | Val Acc: 67.92% ★ [BEST]
Epoch [6/60] Loss: 0.7585 Train Acc: 71.91% | Val Acc: 74.17% ★ [BEST]
Epoch [7/60] Loss: 0.6613 Train Acc: 75.78% | Val Acc: 80.70% ★ [BEST]
Epoch [8/60] Loss: 0.5963 Train Acc: 78.56% | Val Acc: 78.12%
Epoch [9/60] Loss: 0.5157 Train Acc: 81.34% | Val Acc: 79.32%
Epoch [10/6

## Expansão de contraste

### Contraste + Adam

In [5]:
!python -u main_squeezenet.py --version 1.0 --preprocessing contrast_expansion --optimizer adam --lr 0.001 --weight-decay {WEIGHT_DECAY} --seed {SEED} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --amp --num-workers 2

Dispositivo: cuda | Arquitetura: SqueezeNet 1.0 (do zero) | Pré-processamento: contrast_expansion | Otimizador: ADAM | Épocas: 60 | Batch size: 32 | LR: 0.001 | Weight decay: 0.0002 | K-fold: desativado | Validação: 10% | AMP: True
Removendo 1 imagem(ns) duplicada(s) do treino que também estão no teste.
Imagens de treinamento: 9788
Imagens de validação:   1088
Imagens de teste final: 2720
Epoch [1/60] Loss: 1.3458 Train Acc: 43.47% | Val Acc: 53.77% ★ [BEST]
Epoch [2/60] Loss: 1.0925 Train Acc: 58.74% | Val Acc: 68.29% ★ [BEST]
Epoch [3/60] Loss: 0.8851 Train Acc: 69.93% | Val Acc: 72.24% ★ [BEST]
Epoch [4/60] Loss: 0.7724 Train Acc: 73.89% | Val Acc: 79.78% ★ [BEST]
Epoch [5/60] Loss: 0.6890 Train Acc: 76.32% | Val Acc: 76.29%
Epoch [6/60] Loss: 0.5791 Train Acc: 78.87% | Val Acc: 79.78%
Epoch [7/60] Loss: 0.5241 Train Acc: 80.83% | Val Acc: 78.86%
Epoch [8/60] Loss: 0.4840 Train Acc: 82.57% | Val Acc: 83.46% ★ [BEST]
Epoch [9/60] Loss: 0.4296 Train Acc: 84.40% | Val Acc: 87.50% ★ [BE

### Contraste + SGD

In [6]:
!python -u main_squeezenet.py --version 1.0 --preprocessing contrast_expansion --optimizer sgd --lr {SGD_LR} --weight-decay {WEIGHT_DECAY} --seed {SEED} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --amp --num-workers 2

Dispositivo: cuda | Arquitetura: SqueezeNet 1.0 (do zero) | Pré-processamento: contrast_expansion | Otimizador: SGD | Épocas: 60 | Batch size: 32 | LR: 0.003 | Weight decay: 0.0002 | K-fold: desativado | Validação: 10% | AMP: True
Removendo 1 imagem(ns) duplicada(s) do treino que também estão no teste.
Imagens de treinamento: 9788
Imagens de validação:   1088
Imagens de teste final: 2720
Epoch [1/60] Loss: 1.4662 Train Acc: 37.08% | Val Acc: 49.17% ★ [BEST]
Epoch [2/60] Loss: 1.1059 Train Acc: 58.09% | Val Acc: 67.28% ★ [BEST]
Epoch [3/60] Loss: 0.9522 Train Acc: 65.20% | Val Acc: 64.25%
Epoch [4/60] Loss: 0.8566 Train Acc: 68.04% | Val Acc: 74.54% ★ [BEST]
Epoch [5/60] Loss: 0.7272 Train Acc: 72.82% | Val Acc: 74.26%
Epoch [6/60] Loss: 0.6885 Train Acc: 74.09% | Val Acc: 78.22% ★ [BEST]
Epoch [7/60] Loss: 0.6226 Train Acc: 76.72% | Val Acc: 81.16% ★ [BEST]
Epoch [8/60] Loss: 0.5769 Train Acc: 78.80% | Val Acc: 71.51%
Epoch [9/60] Loss: 0.4982 Train Acc: 81.79% | Val Acc: 83.82% ★ [BES

## Equalização de histograma

### Equalização + Adam

In [ ]:
!python -u main_squeezenet.py --version 1.0 --preprocessing histogram_equalization --optimizer adam --lr 0.003 --weight-decay {WEIGHT_DECAY} --seed 115 --epochs {EPOCHS} --batch-size {BATCH_SIZE} --amp --num-workers 2

Dispositivo: cuda | Arquitetura: SqueezeNet 1.0 (do zero) | Pré-processamento: histogram_equalization | Otimizador: ADAM | Épocas: 60 | Batch size: 32 | LR: 0.001 | Weight decay: 0 | K-fold: desativado | Validação: 10% | AMP: True
Removendo 1 imagem(ns) duplicada(s) do treino que também estão no teste.
Imagens de treinamento: 9788
Imagens de validação:   1088
Imagens de teste final: 2720
Epoch [1/60] Loss: 1.7712 Train Acc: 20.66% | Val Acc: 20.40% ★ [BEST]
Epoch [2/60] Loss: 1.7729 Train Acc: 19.53% | Val Acc: 20.59% ★ [BEST]
Epoch [3/60] Loss: 1.7591 Train Acc: 20.14% | Val Acc: 20.40%
Epoch [4/60] Loss: 1.7587 Train Acc: 20.13% | Val Acc: 20.40%
Epoch [5/60] Loss: 1.7580 Train Acc: 20.27% | Val Acc: 20.59%
Epoch [6/60] Loss: 1.7582 Train Acc: 20.19% | Val Acc: 20.40%
Epoch [7/60] Loss: 1.7583 Train Acc: 19.45% | Val Acc: 20.59%
Epoch [8/60] Loss: 1.7585 Train Acc: 19.97% | Val Acc: 20.59%
Epoch [9/60] Loss: 1.7581 Train Acc: 20.68% | Val Acc: 20.40%
Epoch [10/60] Loss: 1.7579 Train 

### Equalização + SGD

In [8]:
!python -u main_squeezenet.py --version 1.0 --preprocessing histogram_equalization --optimizer sgd --lr {SGD_LR} --weight-decay {WEIGHT_DECAY} --seed {SEED} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --amp --num-workers 2

Dispositivo: cuda | Arquitetura: SqueezeNet 1.0 (do zero) | Pré-processamento: histogram_equalization | Otimizador: SGD | Épocas: 60 | Batch size: 32 | LR: 0.003 | Weight decay: 0.0002 | K-fold: desativado | Validação: 10% | AMP: True
Removendo 1 imagem(ns) duplicada(s) do treino que também estão no teste.
Imagens de treinamento: 9788
Imagens de validação:   1088
Imagens de teste final: 2720
Epoch [1/60] Loss: 1.7371 Train Acc: 19.91% | Val Acc: 30.33% ★ [BEST]
Epoch [2/60] Loss: 1.7161 Train Acc: 22.39% | Val Acc: 21.32%
Epoch [3/60] Loss: 1.6807 Train Acc: 25.31% | Val Acc: 34.01% ★ [BEST]
Epoch [4/60] Loss: 1.5532 Train Acc: 34.80% | Val Acc: 34.65% ★ [BEST]
Epoch [5/60] Loss: 1.4839 Train Acc: 39.33% | Val Acc: 41.36% ★ [BEST]
Epoch [6/60] Loss: 1.3886 Train Acc: 44.24% | Val Acc: 44.76% ★ [BEST]
Epoch [7/60] Loss: 1.2279 Train Acc: 51.64% | Val Acc: 58.64% ★ [BEST]
Epoch [8/60] Loss: 1.1106 Train Acc: 55.02% | Val Acc: 54.41%
Epoch [9/60] Loss: 1.0028 Train Acc: 60.22% | Val Acc: 

## Resultados

Ao final de cada célula, o script mostra accuracy, precision, recall e F1-score no teste e informa o caminho do checkpoint. Os resultados permanecem no output da célula; os pesos ficam em `results/`. Para repetir um cenário, execute somente a célula correspondente.